In [0]:
from pyspark.sql import functions as F

destinations = spark.table("trip_planner.bronze.destinations_geocoded")
weather = spark.table("trip_planner.bronze.weather_snapshots")

silver_destinations = (
    destinations
    .join(weather, on="destination", how="left")
    .select(
        "destination", "country", "timezone", "lat", "lon",
        "temperature_c", "windspeed", "weather_code", "pm10", "uv_index"
    )
)

silver_destinations.write.mode("overwrite").saveAsTable("trip_planner.silver.destinations")
silver_destinations.show(truncate=False)

In [0]:
descriptions = spark.table("trip_planner.bronze.destination_descriptions")

silver_descriptions = descriptions.withColumn(
    "embedding_text",
    F.concat(
        F.col("title"), F.lit(" — "), F.col("summary")
    )
).filter(F.col("summary").isNotNull())

silver_descriptions.write.mode("overwrite").saveAsTable("trip_planner.silver.destination_descriptions")
silver_descriptions.select("destination", "title", "embedding_text").show(truncate=60)